# PneumoScan -- Full Evaluation & Model Comparison

This notebook runs every saved model through a **uniform evaluation pipeline** on the held-out test set.  
We collect accuracy, macro-F1, AUC-ROC, AUC-PR, Cohen's Kappa, and inference latency, then present:

- A sortable comparison table  
- Grouped bar charts for the key metrics  
- Confusion matrices for every model side-by-side  
- All ROC curves on a single plot  
- Multi-threshold analysis for the best model  

---

## 1 -- Setup

In [ ]:
# === Colab Setup ===
import os, sys

# Mount Google Drive & clone repo (only runs in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    REPO_DIR = '/content/PneumoScan'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/muhammadrakib2299/PneumoScan.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull
    
    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
    print("Running on Google Colab")
except ImportError:
    sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
    print("Running locally")

# === Imports ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import roc_curve, auc

import config
from data_loader import load_test_dataset
from evaluate import (
    evaluate_all_models,
    evaluate_model,
    predict_dataset,
    multi_threshold_analysis,
)
from utils import (
    plot_confusion_matrix,
    plot_confusion_matrix_normalized,
    plot_roc_curves,
    plot_model_comparison,
    save_figure,
)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# Create all required output directories
config.ensure_dirs()

## 2 -- Load Test Dataset

In [ ]:
test_ds = load_test_dataset()
print("Test dataset loaded.")

## 3 -- Evaluate All Saved Models

`evaluate_all_models` iterates over every model listed in `config.MODEL_NAMES`, loads the `.keras` checkpoint, and runs the full evaluation pipeline (metrics + plots).  
Results are collected into a dictionary and also saved to `outputs/reports/model_comparison.csv`.

In [ ]:
all_metrics = evaluate_all_models(test_ds)

## 4 -- Comparison Table

In [ ]:
# Build a clean comparison DataFrame
rows = []
for m in all_metrics.values():
    rows.append({
        "Model": m["model_name"],
        "Accuracy": m["accuracy"],
        "F1 (Macro)": m["f1_macro"],
        "F1 (Weighted)": m["f1_weighted"],
        "AUC-ROC": m["auc_roc"],
        "AUC-PR": m["auc_pr"],
        "Cohen's Kappa": m["cohens_kappa"],
        "Inference (ms)": m["inference_ms"],
    })

comparison_df = pd.DataFrame(rows)
comparison_df = comparison_df.sort_values("AUC-ROC", ascending=False).reset_index(drop=True)

# Pretty-print with 4 decimal places
styled = comparison_df.style.format({
    "Accuracy": "{:.4f}",
    "F1 (Macro)": "{:.4f}",
    "F1 (Weighted)": "{:.4f}",
    "AUC-ROC": "{:.4f}",
    "AUC-PR": "{:.4f}",
    "Cohen's Kappa": "{:.4f}",
    "Inference (ms)": "{:.2f}",
}).background_gradient(subset=["AUC-ROC", "F1 (Macro)"], cmap="YlGn")

display(styled)

In [ ]:
# Save to CSV for reference
csv_path = os.path.join(config.REPORTS_DIR, "model_comparison.csv")
comparison_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

## 5 -- Comparison Bar Charts

Visual comparison of the three headline metrics across all models.

In [ ]:
# Prepare results dict for plot_model_comparison
results_for_plot = {
    m["model_name"]: {
        "accuracy": m["accuracy"],
        "f1_macro": m["f1_macro"],
        "auc_roc": m["auc_roc"],
    }
    for m in all_metrics.values()
}

for metric in ["accuracy", "f1_macro", "auc_roc"]:
    plot_model_comparison(
        results_for_plot,
        metric=metric,
        save_path=os.path.join(config.COMPARISON_DIR, f"comparison_{metric}.png"),
    )

## 6 -- Multi-Threshold Analysis (Best Model)

The best model (highest AUC-ROC) is evaluated at multiple confidence thresholds.  
This shows the **accuracy / F1 vs coverage** trade-off -- useful for deciding a deployment threshold.

In [ ]:
# Identify best model by AUC-ROC
best_name = comparison_df.iloc[0]["Model"]
best_path = config.MODEL_SAVE_PATHS[best_name]

print(f"Best model: {best_name}")
print(f"Loading from: {best_path}")

best_model = keras.models.load_model(best_path)

In [ ]:
threshold_df = multi_threshold_analysis(
    best_model, test_ds, best_name,
    save_path=os.path.join(config.COMPARISON_DIR, f"{best_name}_threshold_analysis.png"),
)
display(threshold_df)

## 7 -- Confusion Matrices (Side-by-Side)

All models' normalised confusion matrices plotted in a single figure for quick visual comparison.

In [ ]:
from sklearn.metrics import confusion_matrix

model_names_eval = list(all_metrics.keys())
n_models = len(model_names_eval)

fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, name in zip(axes, model_names_eval):
    model_path = config.MODEL_SAVE_PATHS[name]
    model = keras.models.load_model(model_path)
    y_true_oh, y_true, y_pred_proba, y_pred = predict_dataset(model, test_ds)

    cm = confusion_matrix(y_true, y_pred, normalize="true")
    sns.heatmap(
        cm, annot=True, fmt=".2%", cmap="Blues",
        xticklabels=config.CLASS_NAMES, yticklabels=config.CLASS_NAMES,
        ax=ax,
    )
    ax.set_title(name, fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

fig.suptitle("Normalised Confusion Matrices -- All Models", fontsize=15, fontweight="bold")
plt.tight_layout()
save_figure(fig, os.path.join(config.COMPARISON_DIR, "all_confusion_matrices.png"))

## 8 -- All ROC Curves on One Plot

We overlay the macro-averaged ROC curve from every model so we can directly compare discriminative ability.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
colors = ["#2196F3", "#F44336", "#4CAF50", "#FF9800", "#9C27B0", "#795548"]

for idx, name in enumerate(model_names_eval):
    model_path = config.MODEL_SAVE_PATHS[name]
    model = keras.models.load_model(model_path)
    y_true_oh, y_true, y_pred_proba, y_pred = predict_dataset(model, test_ds)

    # Macro-average ROC: compute per-class, then average
    all_fpr = np.linspace(0, 1, 200)
    mean_tpr = np.zeros_like(all_fpr)
    for c in range(config.NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_true_oh[:, c], y_pred_proba[:, c])
        mean_tpr += np.interp(all_fpr, fpr, tpr)
    mean_tpr /= config.NUM_CLASSES
    macro_auc = auc(all_fpr, mean_tpr)

    ax.plot(all_fpr, mean_tpr, color=colors[idx % len(colors)], linewidth=2,
            label=f"{name} (macro AUC = {macro_auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.4)
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("Macro-Averaged ROC Curves -- All Models", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, os.path.join(config.COMPARISON_DIR, "all_roc_curves_overlay.png"))

## 9 -- Comprehensive Analysis

### Which Model is Best?

Based on the results above, we can draw several conclusions:

1. **Overall best (AUC-ROC / F1-Macro)**: The table in Section 4 shows the top model -- typically one of the transfer-learning architectures (DenseNet-121 or EfficientNet-B0) which benefit from ImageNet pre-training and strong feature-reuse mechanisms.  
2. **Fastest inference**: MobileNetV2 is designed for edge deployment and consistently offers the lowest per-image latency.  
3. **Custom CNN**: Our from-scratch CNN achieves respectable performance but generally trails behind pre-trained models, confirming the value of transfer learning for medical imaging with limited data.

### Trade-offs to Consider

| Criterion | Recommended Model |
|---|---|
| Maximum accuracy / AUC | DenseNet-121 or EfficientNet-B0 |
| Real-time / edge deployment | MobileNetV2 |
| Best recall on VIRUS class | Check per-class confusion matrices above |
| Interpretability | Custom CNN (simpler gradient flow) |
| Ensemble for production | Top-3 weighted voting |

### Clinical Implications

- **High sensitivity** for BACTERIA / VIRUS is more important than high specificity in a screening context (false negatives can delay treatment).  
- The **multi-threshold analysis** (Section 6) shows that tightening the confidence threshold improves precision at the cost of coverage -- useful for a two-stage workflow where uncertain cases are reviewed by a radiologist.  
- **Cohen's Kappa > 0.8** indicates almost perfect inter-rater agreement between model and ground truth, which is encouraging for clinical deployment.

---

**Next notebook**: `09_explainability_gradcam_lime.ipynb` -- visual explanations for model predictions.